In [ ]:
from widgets.python.env import ATRVEnv
import gymnasium as gym
from gymnasium.wrappers import TimeLimit
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnNoModelImprovement
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, SubprocVecEnv
from typing import Callable, Optional


def make_env(
    render_mode: Optional[str] = None,
    port: int = 0,
    sync_real_time: bool = False,
) -> Callable:
    def _init() -> gym.Env:
        env = ATRVEnv(render_mode, port, sync_real_time)
        env = TimeLimit(env, max_episode_steps=200)
        return env

    return _init

Run each environment on separate processes using SubprocVecEnv

In [2]:
num_envs = 4
train_env = SubprocVecEnv([make_env() for _ in range(num_envs)])
train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

Loading URDF from /home/kane/Dev/src/kane/src/atrv_driving/atrvjr/atrvjr.urdf
Loading URDF from /home/kane/Dev/src/kane/src/atrv_driving/atrvjr/atrvjr.urdf
Loading URDF from /home/kane/Dev/src/kane/src/atrv_driving/atrvjr/atrvjr.urdf
Loading URDF from /home/kane/Dev/src/kane/src/atrv_driving/atrvjr/atrvjr.urdf


In [3]:
model = SAC(
    "MlpPolicy",
    train_env,
)

In [ ]:
# takes around 30 minutes
model.learn(total_timesteps=500000)

In [6]:
# save the normalization statistics
train_env.save("./models/vec_normalize.pkl")
train_env.close()

In [7]:
test_env = DummyVecEnv([make_env(render_mode="human", port=24546, sync_real_time=True)])
test_env = VecNormalize.load("./models/vec_normalize.pkl", test_env)
test_env.training = False
test_env.norm_reward = False

Loading URDF from /home/kane/Dev/src/kane/src/atrv_driving/atrvjr/atrvjr.urdf
[WebUI] Listening at http://newton:24546


In [8]:
terminated = False
obs = test_env.reset()
while not terminated:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, info = test_env.step(action)

In [9]:
test_env.close()